In [9]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Define the dataset
html_data = ["<form>", "<input type='text'", "<input type='password'", "<input type='submit'", "<input type='checkbox'", "<input type='radio'", "<input type='email'", "<input type='button'"]

# Create input and output sequences from the dataset
input_sequences = []
output_sequences = []

for data in html_data:
    for i in range(1, len(data)):
        input_seq = data[:i]
        output_seq = data[i]
        input_sequences.append(input_seq)
        output_sequences.append(output_seq)

# Convert characters to integers
chars = sorted(set(''.join(html_data)))
char_to_int = {char: i for i, char in enumerate(chars)}

# Convert sequences to integers
input_sequences_int = [[char_to_int[char] for char in seq] for seq in input_sequences]
output_sequences_int = [char_to_int[char] for char in output_sequences]

# Pad sequences to ensure uniform length
max_sequence_length = max(len(seq) for seq in input_sequences_int)
input_sequences_int = pad_sequences(input_sequences_int, maxlen=max_sequence_length, padding='pre')
output_sequences_int = np.array(output_sequences_int)

# Define model architecture
model = Sequential([
    LSTM(64, input_shape=(max_sequence_length, 1)),
    Dense(len(chars), activation='softmax')
])

# Compile model
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Reshape input sequences for compatibility with LSTM
input_sequences_int = np.reshape(input_sequences_int, (input_sequences_int.shape[0], input_sequences_int.shape[1], 1))

# Train the model
model.fit(input_sequences_int, output_sequences_int, epochs=100, batch_size=32)

# Save the model
model.save("html_rnn_model.h5")


Epoch 1/100
5/5 [==============================] - 2s 12ms/step - loss: 3.3437 - accuracy: 0.0217
Epoch 2/100
5/5 [==============================] - 0s 11ms/step - loss: 3.2051 - accuracy: 0.0290
Epoch 3/100
5/5 [==============================] - 0s 11ms/step - loss: 3.0986 - accuracy: 0.0580
Epoch 4/100
5/5 [==============================] - 0s 12ms/step - loss: 3.0147 - accuracy: 0.1594
Epoch 5/100
5/5 [==============================] - 0s 11ms/step - loss: 2.9534 - accuracy: 0.1377
Epoch 6/100
5/5 [==============================] - 0s 11ms/step - loss: 2.8996 - accuracy: 0.1014
Epoch 7/100
5/5 [==============================] - 0s 11ms/step - loss: 2.8592 - accuracy: 0.1377
Epoch 8/100
5/5 [==============================] - 0s 11ms/step - loss: 2.8210 - accuracy: 0.1812
Epoch 9/100
5/5 [==============================] - 0s 11ms/step - loss: 2.7926 - accuracy: 0.1884
Epoch 10/100
5/5 [==============================] - 0s 11ms/step - loss: 2.7543 - accuracy: 0.1739
Epoch 11/100
5/5 [=

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [18]:
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load the trained model
model = load_model("html_rnn_model.h5")

# Define function to generate HTML code completion
def complete_html_code(input_str, model, char_to_int, int_to_char, max_len=50):
    result = input_str.lower()

    # Pre-allocate array to store predicted characters
    predicted_chars = np.zeros(max_len)

    for i in range(max_len):
        encoded_input = np.array([[char_to_int.get(char, 0) for char in result]])
        encoded_input = pad_sequences(encoded_input, maxlen=max_len, padding='pre')
        encoded_input = np.reshape(encoded_input, (encoded_input.shape[0], encoded_input.shape[1], 1))
        predicted_char_int = np.argmax(model.predict(encoded_input), axis=-1)
        predicted_chars[i] = predicted_char_int

        # Convert predicted character index to character
        predicted_char = int_to_char[int(predicted_chars[i])]

        if predicted_char == ">":
            break

        result += predicted_char

    return result

# Test the model
input_str = "<form"
completed_html = complete_html_code(input_str, model, char_to_int, {i: char for char, i in char_to_int.items()}, max_len=max_sequence_length)
print("Input:", input_str)
print("Completed HTML:", completed_html)


1/1 [==============================] - 0s 21ms/step


<ipython-input-18-084df2aff4f9>:20: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  predicted_chars[i] = predicted_char_int


1/1 [==============================] - 0s 21ms/step
Input: <form
Completed HTML: <formt type='sasa'io''io''
